# Phase 5: Backtesting, Validation & Research Hygiene  
## Notebook 05_12 — Strategy Tearsheet Generator

### Research question

How do we turn raw backtest, validation, risk, cost, capacity, and monitoring outputs into a reproducible professional strategy tearsheet suitable for research review?

This notebook closes **Phase 5: Backtesting, Validation & Research Hygiene**.

It follows:

```text
05_01_vectorized_backtest_engine
05_02_event_driven_backtest_engine
05_03_transaction_costs_slippage_latency
05_04_walk_forward_optimization
05_05_bayesian_strategy_calibration
05_06_performance_haircut_and_deflated_sharpe
05_07_purged_kfold_and_embargo_cv
05_08_probability_of_backtest_overfitting
05_09_white_reality_check_bootstrap
05_10_turnover_capacity_and_alpha_decay
05_11_live_paper_trading_dashboard_without_execution
05_12_strategy_tearsheet_generator
```

This notebook combines the evidence from Phase 5 into a single reporting layer.

It covers:

1. strategy metadata;
2. input schema validation;
3. NAV and return analytics;
4. performance summary;
5. drawdown table;
6. rolling risk metrics;
7. benchmark-relative metrics;
8. factor attribution;
9. asset and class attribution;
10. turnover and cost report;
11. capacity snapshot;
12. tail-risk report;
13. regime/stress performance;
14. validation evidence summary;
15. overfitting controls summary;
16. live paper monitoring summary;
17. governance scorecard;
18. red/amber/green decision rules;
19. chart generation;
20. Markdown tearsheet export;
21. JSON manifest export;
22. audit checks.

The key idea:

> A strategy is not research-complete until the evidence is organised into a reproducible report that a PM, risk manager, or reviewer can challenge.

## 1. What a professional tearsheet should answer

A useful strategy tearsheet should answer:

1. **What is the strategy?**
2. **What data was used?**
3. **What period was tested?**
4. **What are the headline returns?**
5. **What are the drawdowns and tail losses?**
6. **What exposures created the returns?**
7. **How much turnover and cost did it require?**
8. **Does performance survive validation?**
9. **Does it survive multiple-testing correction?**
10. **Can it scale?**
11. **Did it behave in live paper mode?**
12. **What are the reasons not to trust it?**

A tearsheet is not marketing. It is an evidence pack.

## 2. Reporting philosophy

This notebook deliberately reports both good and bad evidence.

The tearsheet includes:

- performance metrics;
- validation metrics;
- overfitting diagnostics;
- cost and capacity diagnostics;
- governance flags;
- audit checks;
- limitations.

A strong research culture does not hide caveats. It makes caveats easy to review.

## 3. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class TearsheetConfig:
    n_days: int = 1_800
    annualisation: int = 252
    seed: int = 42
    strategy_name: str = "Synthetic Multi-Asset Momentum / Carry Strategy"
    strategy_id: str = "SYNTH_MOM_CARRY_V1"
    benchmark_name: str = "60/25/15 Equity/Bond/Gold Benchmark"
    transaction_cost_bps: float = 4.0
    initial_capital: float = 10_000_000.0
    gross_target: float = 1.20
    max_abs_weight: float = 0.25
    rebalance_step: int = 5
    cvar_alpha: float = 0.95
    rolling_window: int = 63
    max_drawdown_warn: float = -0.15
    max_drawdown_fail: float = -0.25
    min_sharpe_warn: float = 0.50
    min_deflated_sharpe_probability: float = 0.80
    max_pbo: float = 0.40
    max_wrc_pvalue: float = 0.10
    max_cost_drag: float = 0.05
    min_capacity_aum: float = 50_000_000.0
    output_subdir: str = "strategy_tearsheet_generator_v1"

config = TearsheetConfig()
config

## 4. Simulate a complete strategy evidence pack

In a real project, this notebook would read outputs from prior notebooks.

For portability, we simulate a complete evidence pack:

- strategy returns;
- benchmark returns;
- target weights;
- asset returns;
- factor returns;
- cost series;
- validation reports;
- overfitting metrics;
- live paper dashboard metrics.

In [ ]:
def simulate_tearsheet_inputs(config: TearsheetConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2017-01-01", periods=config.n_days)

    assets = [
        "US_EQ", "EU_EQ", "EM_EQ",
        "US_BOND", "EU_BOND",
        "GOLD", "OIL", "COPPER",
        "FX_CARRY", "TREND",
        "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "EM_EQ": "equity",
        "US_BOND": "bond",
        "EU_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "COPPER": "commodity",
        "FX_CARRY": "fx",
        "TREND": "alternative",
        "REIT": "real_asset",
        "BTC_PROXY": "crypto",
    }

    factor_names = ["market", "rates", "commodity", "carry", "trend", "crypto"]
    factor_vol_daily = np.array([0.010, 0.004, 0.012, 0.006, 0.007, 0.035])
    factor_corr = np.array([
        [1.00, -0.25, 0.25, 0.15, -0.15, 0.35],
        [-0.25, 1.00, 0.10, -0.05, 0.15, -0.20],
        [0.25, 0.10, 1.00, 0.10, 0.05, 0.20],
        [0.15, -0.05, 0.10, 1.00, 0.00, 0.15],
        [-0.15, 0.15, 0.05, 0.00, 1.00, 0.00],
        [0.35, -0.20, 0.20, 0.15, 0.00, 1.00],
    ])

    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)
    loadings.loc[["US_EQ", "EU_EQ", "EM_EQ", "REIT"], "market"] = [1.00, 0.95, 1.20, 0.70]
    loadings.loc[["US_BOND", "EU_BOND"], "rates"] = [1.00, 0.90]
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc["FX_CARRY", "carry"] = 1.00
    loadings.loc["TREND", "trend"] = 1.00
    loadings.loc["TREND", "market"] = -0.25
    loadings.loc["BTC_PROXY", "crypto"] = 1.00
    loadings.loc["BTC_PROXY", "market"] = 0.40
    loadings.loc["GOLD", "rates"] = 0.25
    loadings.loc["REIT", "rates"] = -0.30

    ann_mu = pd.Series({
        "US_EQ": 0.070, "EU_EQ": 0.060, "EM_EQ": 0.080,
        "US_BOND": 0.025, "EU_BOND": 0.020,
        "GOLD": 0.035, "OIL": 0.045, "COPPER": 0.040,
        "FX_CARRY": 0.030, "TREND": 0.050,
        "REIT": 0.060, "BTC_PROXY": 0.120,
    })

    ann_vol = pd.Series({
        "US_EQ": 0.16, "EU_EQ": 0.18, "EM_EQ": 0.25,
        "US_BOND": 0.06, "EU_BOND": 0.07,
        "GOLD": 0.18, "OIL": 0.32, "COPPER": 0.28,
        "FX_CARRY": 0.10, "TREND": 0.12,
        "REIT": 0.20, "BTC_PROXY": 0.70,
    })

    adv_usd = pd.Series({
        "US_EQ": 9e9, "EU_EQ": 4e9, "EM_EQ": 1.5e9,
        "US_BOND": 12e9, "EU_BOND": 6e9,
        "GOLD": 3e9, "OIL": 2.5e9, "COPPER": 1e9,
        "FX_CARRY": 2.5e9, "TREND": 0.7e9,
        "REIT": 1.5e9, "BTC_PROXY": 0.8e9,
    })

    regime = np.zeros(config.n_days, dtype=int)
    stress_type = np.array(["normal"] * config.n_days, dtype=object)
    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    asset_returns = np.zeros((config.n_days, len(assets)))
    factor_returns = np.zeros((config.n_days, len(factor_names)))

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.4
        f = rng.multivariate_normal(np.zeros(len(factor_names)), factor_cov * vol_multiplier**2)

        u = rng.uniform()
        if u < 0.008:
            stress_type[t] = "equity_crash"
            f[0] -= rng.uniform(0.035, 0.100)
            f[1] += rng.uniform(0.004, 0.025)
            f[5] -= rng.uniform(0.080, 0.220)
            f[4] += rng.uniform(0.005, 0.040)
        elif u < 0.014:
            stress_type[t] = "inflation_shock"
            f[1] -= rng.uniform(0.010, 0.035)
            f[2] += rng.uniform(0.030, 0.090)
            f[0] -= rng.uniform(0.010, 0.050)
        elif u < 0.020:
            stress_type[t] = "commodity_crash"
            f[2] -= rng.uniform(0.040, 0.120)
            f[0] -= rng.uniform(0.005, 0.030)

        eps = rng.standard_t(df=6, size=len(assets)) * np.sqrt((6 - 2) / 6)
        eps = eps * ann_vol.values / np.sqrt(config.annualisation) * 0.35 * vol_multiplier

        asset_returns[t] = ann_mu.values / config.annualisation + loadings.to_numpy() @ f + eps
        factor_returns[t] = f

    asset_returns = pd.DataFrame(asset_returns, index=dates, columns=assets)
    prices = 100 * np.exp(np.log1p(asset_returns).cumsum())
    factor_returns = pd.DataFrame(factor_returns, index=dates, columns=factor_names)

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "ann_mu_assumption": [ann_mu[a] for a in assets],
        "ann_vol_assumption": [ann_vol[a] for a in assets],
        "adv_usd": [adv_usd[a] for a in assets],
    })

    regime_info = pd.DataFrame({
        "regime": regime,
        "stress_type": stress_type,
    }, index=dates)

    benchmark_returns = 0.60 * asset_returns["US_EQ"] + 0.25 * asset_returns["US_BOND"] + 0.15 * asset_returns["GOLD"]

    return asset_returns, prices, factor_returns, metadata, regime_info, benchmark_returns

asset_returns, prices, factor_returns, metadata, regime_info, benchmark_returns = simulate_tearsheet_inputs(config)
assets = metadata["asset"].tolist()

asset_returns.head()

## 5. Build a strategy from signals

In [ ]:
def cross_sectional_zscore(df):
    mu = df.mean(axis=1)
    sigma = df.std(axis=1).replace(0.0, np.nan)
    return df.sub(mu, axis=0).div(sigma, axis=0).fillna(0.0)

def build_signal(prices, asset_returns):
    momentum_63 = cross_sectional_zscore(prices.pct_change(63))
    momentum_126 = cross_sectional_zscore(prices.pct_change(126))
    reversal_5 = cross_sectional_zscore(-prices.pct_change(5))
    vol_63 = asset_returns.rolling(63).std() * np.sqrt(config.annualisation)
    low_vol = cross_sectional_zscore(-vol_63)

    signal = 0.45 * momentum_126 + 0.25 * momentum_63 + 0.15 * reversal_5 + 0.15 * low_vol
    return signal.clip(-3, 3).fillna(0.0)

def signal_to_weights(signal, asset_returns, config):
    vol = asset_returns.rolling(63).std().shift(1)
    vol = vol.fillna(asset_returns.expanding().std().shift(1)).fillna(asset_returns.std())
    inv_vol = 1.0 / vol.replace(0.0, np.nan)

    raw = signal.sub(signal.mean(axis=1), axis=0).mul(inv_vol, axis=0)
    weights = raw.div(raw.abs().sum(axis=1).replace(0.0, np.nan), axis=0).mul(config.gross_target).fillna(0.0)

    weights = weights.clip(-config.max_abs_weight, config.max_abs_weight)
    gross = weights.abs().sum(axis=1).replace(0.0, np.nan)
    weights = weights.div(gross, axis=0).mul(config.gross_target).fillna(0.0)

    return weights

def apply_rebalance(weights, step):
    scheduled = weights.copy() * np.nan
    scheduled.iloc[::step] = weights.iloc[::step]
    return scheduled.ffill().fillna(0.0)

signal = build_signal(prices, asset_returns)
target_weights = apply_rebalance(signal_to_weights(signal, asset_returns, config), config.rebalance_step)
held_weights = target_weights.shift(1).fillna(0.0)
delta_weights = held_weights.diff().fillna(held_weights)

target_weights.tail()

## 6. Build backtest accounting series

In [ ]:
def run_strategy_accounting(asset_returns, held_weights, delta_weights, benchmark_returns, config):
    gross_return = (held_weights * asset_returns).sum(axis=1)
    turnover = delta_weights.abs().sum(axis=1)
    transaction_cost = turnover * config.transaction_cost_bps / 10000.0
    net_return = gross_return - transaction_cost

    strategy_nav = (1 + net_return).cumprod()
    benchmark_nav = (1 + benchmark_returns).cumprod()

    accounting = pd.DataFrame({
        "gross_return": gross_return,
        "transaction_cost": transaction_cost,
        "net_return": net_return,
        "benchmark_return": benchmark_returns,
        "active_return": net_return - benchmark_returns,
        "strategy_nav": strategy_nav,
        "benchmark_nav": benchmark_nav,
        "turnover": turnover,
        "gross_exposure": held_weights.abs().sum(axis=1),
        "net_exposure": held_weights.sum(axis=1),
        "long_exposure": held_weights.clip(lower=0.0).sum(axis=1),
        "short_exposure": held_weights.clip(upper=0.0).sum(axis=1),
    }, index=asset_returns.index)

    return accounting

accounting = run_strategy_accounting(asset_returns, held_weights, delta_weights, benchmark_returns, config)

accounting.head()

## 7. Input schema validation

In [ ]:
def validate_tearsheet_inputs(accounting, asset_returns, held_weights, factor_returns, metadata):
    checks = []

    required_accounting = [
        "net_return", "benchmark_return", "strategy_nav", "benchmark_nav",
        "turnover", "gross_exposure", "net_exposure"
    ]

    for col in required_accounting:
        checks.append({
            "check": f"accounting_has_{col}",
            "passed": col in accounting.columns,
            "detail": col,
        })

    checks.append({
        "check": "accounting_index_unique",
        "passed": bool(accounting.index.is_unique),
        "detail": f"{accounting.index.min()} to {accounting.index.max()}",
    })

    checks.append({
        "check": "asset_returns_align_with_weights",
        "passed": bool(asset_returns.index.equals(held_weights.index) and list(asset_returns.columns) == list(held_weights.columns)),
        "detail": f"{len(asset_returns)} rows, {len(asset_returns.columns)} assets",
    })

    checks.append({
        "check": "factor_returns_align_dates",
        "passed": bool(factor_returns.index.equals(accounting.index)),
        "detail": f"{len(factor_returns)} factor rows",
    })

    checks.append({
        "check": "metadata_covers_assets",
        "passed": bool(set(asset_returns.columns).issubset(set(metadata["asset"]))),
        "detail": f"{metadata['asset'].nunique()} metadata assets",
    })

    checks.append({
        "check": "numeric_outputs_finite",
        "passed": bool(np.isfinite(accounting.select_dtypes(include=[float, int]).to_numpy()).all()),
        "detail": "accounting numeric matrix",
    })

    return pd.DataFrame(checks)

input_validation = validate_tearsheet_inputs(accounting, asset_returns, held_weights, factor_returns, metadata)

input_validation

## 8. Core performance metrics

In [ ]:
def max_drawdown(nav):
    dd = nav / nav.cummax() - 1.0
    return dd, float(dd.min())

def historical_var_cvar(losses, alpha):
    losses = pd.Series(losses).dropna()
    if len(losses) == 0:
        return np.nan, np.nan
    var = losses.quantile(alpha)
    tail = losses[losses >= var]
    return float(var), float(tail.mean()) if len(tail) else np.nan

def alpha_beta(strategy_returns, benchmark_returns, annualisation):
    joined = pd.concat([strategy_returns.rename("strategy"), benchmark_returns.rename("benchmark")], axis=1).dropna()
    if len(joined) < 5 or joined["benchmark"].var(ddof=1) == 0:
        return np.nan, np.nan, np.nan
    beta = joined["strategy"].cov(joined["benchmark"]) / joined["benchmark"].var(ddof=1)
    alpha_daily = joined["strategy"].mean() - beta * joined["benchmark"].mean()
    corr = joined["strategy"].corr(joined["benchmark"])
    return float(alpha_daily * annualisation), float(beta), float(corr)

def performance_snapshot(return_series, nav_series, benchmark_returns, config):
    r = pd.Series(return_series).dropna()
    nav = pd.Series(nav_series).reindex(r.index).dropna()
    _, mdd = max_drawdown(nav)

    ann_return = (1 + r).prod() ** (config.annualisation / len(r)) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(config.annualisation)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = downside.std(ddof=1) * np.sqrt(config.annualisation)
    sortino = ann_return / downside_vol if downside_vol > 0 else np.nan

    calmar = ann_return / abs(mdd) if mdd < 0 else np.nan
    var, cvar = historical_var_cvar(-r, config.cvar_alpha)

    alpha_ann, beta, corr = alpha_beta(r, benchmark_returns.reindex(r.index), config.annualisation)

    return {
        "n_days": len(r),
        "annualised_return": ann_return,
        "annualised_vol": ann_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "calmar": calmar,
        "max_drawdown": mdd,
        "VaR": var,
        "CVaR": cvar,
        "hit_rate": (r > 0).mean(),
        "skew": r.skew(),
        "excess_kurtosis": r.kurtosis(),
        "total_return": (1 + r).prod() - 1,
        "alpha_ann_vs_benchmark": alpha_ann,
        "beta_vs_benchmark": beta,
        "corr_vs_benchmark": corr,
    }

strategy_metrics = performance_snapshot(accounting["net_return"], accounting["strategy_nav"], accounting["benchmark_return"], config)
benchmark_metrics = performance_snapshot(accounting["benchmark_return"], accounting["benchmark_nav"], accounting["benchmark_return"], config)

performance_summary = pd.DataFrame([
    {"series": "strategy", **strategy_metrics},
    {"series": "benchmark", **benchmark_metrics},
])

performance_summary.T

## 9. Drawdown table

In [ ]:
def drawdown_events(nav, top_n=10):
    nav = pd.Series(nav).dropna()
    dd = nav / nav.cummax() - 1.0

    rows = []
    in_drawdown = False
    start = None
    trough = None
    trough_dd = 0.0

    for date, value in dd.items():
        if value < 0 and not in_drawdown:
            in_drawdown = True
            start = date
            trough = date
            trough_dd = value
        elif value < 0 and in_drawdown:
            if value < trough_dd:
                trough = date
                trough_dd = value
        elif value == 0 and in_drawdown:
            rows.append({
                "start": start,
                "trough": trough,
                "recovery": date,
                "max_drawdown": trough_dd,
                "days_to_trough": (trough - start).days,
                "days_to_recovery": (date - start).days,
            })
            in_drawdown = False

    if in_drawdown:
        rows.append({
            "start": start,
            "trough": trough,
            "recovery": pd.NaT,
            "max_drawdown": trough_dd,
            "days_to_trough": (trough - start).days,
            "days_to_recovery": np.nan,
        })

    if len(rows) == 0:
        return pd.DataFrame(columns=["start", "trough", "recovery", "max_drawdown", "days_to_trough", "days_to_recovery"])

    return pd.DataFrame(rows).sort_values("max_drawdown").head(top_n)

drawdown_table = drawdown_events(accounting["strategy_nav"], top_n=10)

drawdown_table

## 10. Rolling metrics

In [ ]:
def rolling_metrics(accounting, config):
    r = accounting["net_return"]
    b = accounting["benchmark_return"]
    active = accounting["active_return"]

    roll = pd.DataFrame(index=accounting.index)
    roll["rolling_return"] = r.rolling(config.rolling_window).mean() * config.annualisation
    roll["rolling_vol"] = r.rolling(config.rolling_window).std() * np.sqrt(config.annualisation)
    roll["rolling_sharpe"] = roll["rolling_return"] / roll["rolling_vol"]
    roll["rolling_active_return"] = active.rolling(config.rolling_window).mean() * config.annualisation
    roll["rolling_active_vol"] = active.rolling(config.rolling_window).std() * np.sqrt(config.annualisation)
    roll["rolling_information_ratio"] = roll["rolling_active_return"] / roll["rolling_active_vol"]
    roll["rolling_beta"] = r.rolling(config.rolling_window).cov(b) / b.rolling(config.rolling_window).var()
    roll["rolling_turnover"] = accounting["turnover"].rolling(config.rolling_window).mean()
    roll["rolling_cost_drag"] = accounting["transaction_cost"].rolling(config.rolling_window).mean() * config.annualisation

    strategy_dd, _ = max_drawdown(accounting["strategy_nav"])
    roll["drawdown"] = strategy_dd

    return roll

rolling_report = rolling_metrics(accounting, config)

rolling_report.tail()

## 11. Factor attribution

In [ ]:
def factor_attribution(strategy_returns, factor_returns, annualisation):
    y = strategy_returns.reindex(factor_returns.index).dropna()
    X = factor_returns.reindex(y.index).dropna()
    y = y.reindex(X.index)

    X_mat = np.column_stack([np.ones(len(X)), X.to_numpy()])
    beta = np.linalg.pinv(X_mat.T @ X_mat) @ X_mat.T @ y.to_numpy()

    fitted = X_mat @ beta
    resid = y.to_numpy() - fitted

    ss_res = float(resid @ resid)
    ss_tot = float(((y.to_numpy() - y.mean()) @ (y.to_numpy() - y.mean())))
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    rows = [{"factor": "alpha", "coefficient": beta[0], "annualised_contribution_proxy": beta[0] * annualisation}]
    for factor, coeff in zip(X.columns, beta[1:]):
        rows.append({
            "factor": factor,
            "coefficient": coeff,
            "annualised_contribution_proxy": coeff * X[factor].mean() * annualisation,
        })

    report = pd.DataFrame(rows)
    summary = pd.DataFrame([{
        "factor_model_r_squared": r2,
        "residual_vol_ann": np.std(resid, ddof=1) * np.sqrt(annualisation),
        "alpha_ann": beta[0] * annualisation,
    }])

    fitted_series = pd.Series(fitted, index=y.index, name="factor_fitted_return")
    residual_series = pd.Series(resid, index=y.index, name="factor_residual_return")

    return report, summary, fitted_series, residual_series

factor_exposure_report, factor_summary, factor_fitted, factor_residual = factor_attribution(
    accounting["net_return"],
    factor_returns,
    config.annualisation,
)

factor_exposure_report, factor_summary

## 12. Asset and class attribution

In [ ]:
def asset_class_attribution(asset_returns, held_weights, metadata, config):
    contribution = held_weights * asset_returns

    asset_rows = []
    for asset in asset_returns.columns:
        asset_rows.append({
            "asset": asset,
            "annualised_contribution": contribution[asset].mean() * config.annualisation,
            "total_contribution": contribution[asset].sum(),
            "contribution_vol_ann": contribution[asset].std(ddof=1) * np.sqrt(config.annualisation),
            "average_weight": held_weights[asset].mean(),
            "average_abs_weight": held_weights[asset].abs().mean(),
        })

    asset_attr = pd.DataFrame(asset_rows).merge(metadata[["asset", "asset_class", "adv_usd"]], on="asset", how="left")

    class_attr = (
        asset_attr
        .groupby("asset_class")
        .agg(
            annualised_contribution=("annualised_contribution", "sum"),
            total_contribution=("total_contribution", "sum"),
            average_abs_weight=("average_abs_weight", "sum"),
        )
        .reset_index()
        .sort_values("annualised_contribution", ascending=False)
    )

    return asset_attr.sort_values("annualised_contribution", ascending=False), class_attr

asset_attribution, class_attribution = asset_class_attribution(asset_returns, held_weights, metadata, config)

asset_attribution.head(10), class_attribution

## 13. Cost, turnover, and capacity snapshot

In [ ]:
def cost_capacity_snapshot(accounting, delta_weights, metadata, config):
    traded_notional = delta_weights.abs() * config.initial_capital
    adv = metadata.set_index("asset")["adv_usd"].reindex(delta_weights.columns)
    participation = traded_notional.div(adv, axis=1)

    rows = []
    for asset in delta_weights.columns:
        rows.append({
            "asset": asset,
            "mean_participation": participation[asset].mean(),
            "p95_participation": participation[asset].quantile(0.95),
            "max_participation": participation[asset].max(),
            "mean_traded_notional": traded_notional[asset].mean(),
            "p95_traded_notional": traded_notional[asset].quantile(0.95),
        })

    capacity_by_asset = pd.DataFrame(rows).merge(metadata[["asset", "asset_class", "adv_usd"]], on="asset", how="left")

    summary = pd.DataFrame([{
        "mean_daily_turnover": accounting["turnover"].mean(),
        "annualised_turnover": accounting["turnover"].mean() * config.annualisation,
        "annualised_cost_drag": accounting["transaction_cost"].mean() * config.annualisation,
        "mean_gross_exposure": accounting["gross_exposure"].mean(),
        "mean_net_exposure": accounting["net_exposure"].mean(),
        "max_turnover": accounting["turnover"].max(),
        "p95_participation_max_asset": capacity_by_asset["p95_participation"].max(),
        "estimated_capacity_proxy_aum": config.initial_capital * 0.10 / max(capacity_by_asset["p95_participation"].max(), 1e-12),
    }])

    return summary, capacity_by_asset.sort_values("p95_participation", ascending=False)

cost_capacity_summary, capacity_by_asset = cost_capacity_snapshot(accounting, delta_weights, metadata, config)

cost_capacity_summary.T, capacity_by_asset.head(10)

## 14. Regime and stress performance

In [ ]:
def regime_performance(accounting, regime_info, config):
    joined = accounting.join(regime_info, how="left")
    rows = []

    for bucket, group in joined.groupby("stress_type"):
        if len(group) < 5:
            continue
        metrics = performance_snapshot(group["net_return"], (1 + group["net_return"]).cumprod(), group["benchmark_return"], config)
        rows.append({"bucket_type": "stress_type", "bucket": bucket, **metrics})

    for bucket, group in joined.groupby("regime"):
        if len(group) < 5:
            continue
        metrics = performance_snapshot(group["net_return"], (1 + group["net_return"]).cumprod(), group["benchmark_return"], config)
        rows.append({"bucket_type": "regime", "bucket": f"regime_{bucket}", **metrics})

    return pd.DataFrame(rows)

regime_report = regime_performance(accounting, regime_info, config)

regime_report

## 15. Validation evidence summary

In [ ]:
def simulate_validation_evidence(config):
    validation = pd.DataFrame([{
        "walk_forward_sharpe": 0.82,
        "walk_forward_max_drawdown": -0.118,
        "bayesian_prob_positive_alpha": 0.71,
        "bayesian_prob_sharpe_gt_0_5": 0.58,
        "deflated_sharpe_probability": 0.83,
        "pbo": 0.34,
        "white_reality_check_pvalue": 0.082,
        "purged_cv_auc": 0.535,
        "purged_cv_embargo_pct": 0.02,
        "live_paper_sharpe": 0.62,
        "live_paper_drawdown": -0.061,
        "live_paper_data_health_pass_rate": 0.97,
        "capacity_estimate_aum": 85_000_000.0,
        "estimated_alpha_half_life_days": 18.5,
    }])

    validation_notes = pd.DataFrame([
        {"area": "walk_forward", "status": "PASS", "note": "Positive out-of-sample Sharpe after costs."},
        {"area": "bayesian", "status": "AMBER", "note": "Posterior alpha probability positive but not overwhelming."},
        {"area": "deflated_sharpe", "status": "PASS", "note": "Deflated Sharpe probability above internal threshold."},
        {"area": "pbo", "status": "PASS", "note": "PBO below warning threshold."},
        {"area": "white_reality_check", "status": "AMBER", "note": "Reality Check p-value is suggestive but not below 5%."},
        {"area": "purged_cv", "status": "AMBER", "note": "Purged CV AUC modest but above random."},
        {"area": "live_paper", "status": "PASS", "note": "Paper mode stable with good data-health pass rate."},
        {"area": "capacity", "status": "PASS", "note": "Estimated capacity exceeds minimum target."},
    ])

    return validation, validation_notes

validation_summary, validation_notes = simulate_validation_evidence(config)

validation_summary.T, validation_notes

## 16. Governance scorecard

In [ ]:
def build_governance_scorecard(performance_summary, cost_capacity_summary, validation_summary, config):
    perf = performance_summary[performance_summary["series"] == "strategy"].iloc[0]
    cost = cost_capacity_summary.iloc[0]
    val = validation_summary.iloc[0]

    rows = []

    def add_check(category, check, value, status, threshold, note):
        rows.append({
            "category": category,
            "check": check,
            "value": value,
            "threshold": threshold,
            "status": status,
            "note": note,
        })

    add_check("performance", "Sharpe above warning threshold", perf["sharpe"], "PASS" if perf["sharpe"] >= config.min_sharpe_warn else "FAIL", config.min_sharpe_warn, "Headline risk-adjusted return.")
    add_check("risk", "Max drawdown within fail threshold", perf["max_drawdown"], "PASS" if perf["max_drawdown"] > config.max_drawdown_fail else "FAIL", config.max_drawdown_fail, "Absolute drawdown guardrail.")
    add_check("risk", "Max drawdown within warning threshold", perf["max_drawdown"], "PASS" if perf["max_drawdown"] > config.max_drawdown_warn else "AMBER", config.max_drawdown_warn, "Warning drawdown guardrail.")
    add_check("cost", "Annualised cost drag acceptable", cost["annualised_cost_drag"], "PASS" if cost["annualised_cost_drag"] <= config.max_cost_drag else "FAIL", config.max_cost_drag, "Turnover and estimated cost pressure.")
    add_check("capacity", "Capacity exceeds minimum AUM", val["capacity_estimate_aum"], "PASS" if val["capacity_estimate_aum"] >= config.min_capacity_aum else "FAIL", config.min_capacity_aum, "Scalability check.")
    add_check("validation", "Deflated Sharpe probability", val["deflated_sharpe_probability"], "PASS" if val["deflated_sharpe_probability"] >= config.min_deflated_sharpe_probability else "AMBER", config.min_deflated_sharpe_probability, "Multiple-testing-adjusted confidence.")
    add_check("validation", "PBO below maximum", val["pbo"], "PASS" if val["pbo"] <= config.max_pbo else "FAIL", config.max_pbo, "Selection-overfit diagnostic.")
    add_check("validation", "White Reality Check p-value", val["white_reality_check_pvalue"], "PASS" if val["white_reality_check_pvalue"] <= config.max_wrc_pvalue else "AMBER", config.max_wrc_pvalue, "Data-snooping adjusted p-value.")
    add_check("live_paper", "Live paper data health", val["live_paper_data_health_pass_rate"], "PASS" if val["live_paper_data_health_pass_rate"] >= 0.95 else "FAIL", 0.95, "Operational stability in shadow mode.")

    scorecard = pd.DataFrame(rows)

    if (scorecard["status"] == "FAIL").any():
        overall = "RED_REJECT_OR_REWORK"
    elif (scorecard["status"] == "AMBER").any():
        overall = "AMBER_REVIEW_REQUIRED"
    else:
        overall = "GREEN_ELIGIBLE_FOR_REVIEW"

    decision = pd.DataFrame([{
        "overall_status": overall,
        "n_pass": int((scorecard["status"] == "PASS").sum()),
        "n_amber": int((scorecard["status"] == "AMBER").sum()),
        "n_fail": int((scorecard["status"] == "FAIL").sum()),
        "can_trade": False,
        "note": "This is a research review status, not trading approval.",
    }])

    return scorecard, decision

governance_scorecard, governance_decision = build_governance_scorecard(
    performance_summary,
    cost_capacity_summary,
    validation_summary,
    config,
)

governance_scorecard, governance_decision

## 17. Chart generation

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
chart_dir = output_dir / "charts"
chart_dir.mkdir(parents=True, exist_ok=True)

def save_nav_chart(accounting, chart_dir):
    path = chart_dir / "nav_comparison.png"
    plt.figure(figsize=(12, 6))
    plt.plot(accounting.index, accounting["strategy_nav"], label="Strategy")
    plt.plot(accounting.index, accounting["benchmark_nav"], label="Benchmark")
    plt.title("Strategy NAV vs Benchmark")
    plt.xlabel("Date")
    plt.ylabel("Growth of 1")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    return path

def save_drawdown_chart(accounting, chart_dir):
    path = chart_dir / "drawdown.png"
    dd, _ = max_drawdown(accounting["strategy_nav"])
    plt.figure(figsize=(12, 5))
    plt.plot(dd.index, dd)
    plt.title("Strategy Drawdown")
    plt.xlabel("Date")
    plt.ylabel("Drawdown")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    return path

def save_rolling_sharpe_chart(rolling_report, chart_dir):
    path = chart_dir / "rolling_sharpe.png"
    plt.figure(figsize=(12, 5))
    plt.plot(rolling_report.index, rolling_report["rolling_sharpe"])
    plt.axhline(0, linestyle="--")
    plt.title("Rolling Sharpe")
    plt.xlabel("Date")
    plt.ylabel("Rolling Sharpe")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    return path

def save_asset_attribution_chart(asset_attribution, chart_dir):
    path = chart_dir / "asset_attribution.png"
    plot_df = asset_attribution.sort_values("annualised_contribution")
    plt.figure(figsize=(10, 6))
    plt.barh(plot_df["asset"], plot_df["annualised_contribution"])
    plt.axvline(0, linestyle="--")
    plt.title("Annualised Contribution by Asset")
    plt.xlabel("Annualised contribution")
    plt.ylabel("Asset")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    return path

def save_factor_exposure_chart(factor_exposure_report, chart_dir):
    path = chart_dir / "factor_exposure.png"
    plot_df = factor_exposure_report[factor_exposure_report["factor"] != "alpha"].copy()
    plt.figure(figsize=(10, 6))
    plt.barh(plot_df["factor"], plot_df["coefficient"])
    plt.axvline(0, linestyle="--")
    plt.title("Factor Exposures")
    plt.xlabel("Coefficient")
    plt.ylabel("Factor")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    return path

def save_cost_capacity_chart(capacity_by_asset, chart_dir):
    path = chart_dir / "capacity_by_asset.png"
    plot_df = capacity_by_asset.sort_values("p95_participation")
    plt.figure(figsize=(10, 6))
    plt.barh(plot_df["asset"], plot_df["p95_participation"])
    plt.title("p95 Participation by Asset")
    plt.xlabel("p95 participation")
    plt.ylabel("Asset")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    return path

chart_paths = {
    "nav_comparison": save_nav_chart(accounting, chart_dir),
    "drawdown": save_drawdown_chart(accounting, chart_dir),
    "rolling_sharpe": save_rolling_sharpe_chart(rolling_report, chart_dir),
    "asset_attribution": save_asset_attribution_chart(asset_attribution, chart_dir),
    "factor_exposure": save_factor_exposure_chart(factor_exposure_report, chart_dir),
    "capacity_by_asset": save_cost_capacity_chart(capacity_by_asset, chart_dir),
}

chart_paths

## 18. Markdown table helpers

In [ ]:
def df_to_markdown_safe(df, max_rows=20, floatfmt=".4f"):
    display = df.copy()
    if len(display) > max_rows:
        display = display.head(max_rows)

    for col in display.columns:
        if pd.api.types.is_float_dtype(display[col]):
            display[col] = display[col].map(lambda x: f"{x:{floatfmt}}" if pd.notna(x) else "")

    return display.to_markdown(index=False)

def format_pct(x):
    if pd.isna(x):
        return "n/a"
    return f"{x:.2%}"

def format_num(x):
    if pd.isna(x):
        return "n/a"
    return f"{x:.3f}"

def format_money(x):
    if pd.isna(x):
        return "n/a"
    return f"${x:,.0f}"

## 19. Generate the strategy tearsheet Markdown

In [ ]:
def generate_markdown_tearsheet(
    config,
    performance_summary,
    drawdown_table,
    rolling_report,
    factor_exposure_report,
    factor_summary,
    asset_attribution,
    class_attribution,
    cost_capacity_summary,
    capacity_by_asset,
    regime_report,
    validation_summary,
    validation_notes,
    governance_scorecard,
    governance_decision,
    input_validation,
    chart_paths,
):
    strategy = performance_summary[performance_summary["series"] == "strategy"].iloc[0]
    benchmark = performance_summary[performance_summary["series"] == "benchmark"].iloc[0]
    cost = cost_capacity_summary.iloc[0]
    val = validation_summary.iloc[0]
    decision = governance_decision.iloc[0]
    rel_chart_paths = {k: str(v) for k, v in chart_paths.items()}

    lines = []
    lines.append(f"# Strategy Tearsheet — {config.strategy_name}")
    lines.append("")
    lines.append(f"**Strategy ID:** `{config.strategy_id}`  ")
    lines.append(f"**Benchmark:** {config.benchmark_name}  ")
    lines.append(f"**Report status:** **{decision['overall_status']}**  ")
    lines.append("**Trading approval:** **No. Research review only.**")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## 1. Executive Summary")
    lines.append("")
    lines.append("| Item | Value |")
    lines.append("|---|---:|")
    lines.append(f"| Annualised return | {format_pct(strategy['annualised_return'])} |")
    lines.append(f"| Annualised volatility | {format_pct(strategy['annualised_vol'])} |")
    lines.append(f"| Sharpe | {format_num(strategy['sharpe'])} |")
    lines.append(f"| Sortino | {format_num(strategy['sortino'])} |")
    lines.append(f"| Max drawdown | {format_pct(strategy['max_drawdown'])} |")
    lines.append(f"| CVaR {int(config.cvar_alpha * 100)}% | {format_pct(strategy['CVaR'])} |")
    lines.append(f"| Alpha vs benchmark | {format_pct(strategy['alpha_ann_vs_benchmark'])} |")
    lines.append(f"| Beta vs benchmark | {format_num(strategy['beta_vs_benchmark'])} |")
    lines.append(f"| Annualised cost drag | {format_pct(cost['annualised_cost_drag'])} |")
    lines.append(f"| Estimated capacity proxy | {format_money(cost['estimated_capacity_proxy_aum'])} |")
    lines.append("")
    lines.append(f"**Review decision:** {decision['overall_status']}  ")
    lines.append(f"**Reason:** {decision['note']}")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## 2. NAV and Drawdown")
    lines.append("")
    lines.append(f"![NAV comparison]({rel_chart_paths['nav_comparison']})")
    lines.append("")
    lines.append(f"![Drawdown]({rel_chart_paths['drawdown']})")
    lines.append("")
    lines.append("## 3. Performance Summary")
    lines.append("")
    lines.append(df_to_markdown_safe(performance_summary, max_rows=10))
    lines.append("")
    lines.append("## 4. Top Drawdowns")
    lines.append("")
    lines.append(df_to_markdown_safe(drawdown_table, max_rows=10))
    lines.append("")
    lines.append("## 5. Rolling Risk")
    lines.append("")
    lines.append(f"![Rolling Sharpe]({rel_chart_paths['rolling_sharpe']})")
    lines.append("")
    lines.append("Latest rolling metrics:")
    lines.append("")
    lines.append(df_to_markdown_safe(rolling_report.tail(5).reset_index().rename(columns={'index': 'date'}), max_rows=5))
    lines.append("")
    lines.append("## 6. Factor Attribution")
    lines.append("")
    lines.append("Factor model summary:")
    lines.append("")
    lines.append(df_to_markdown_safe(factor_summary, max_rows=5))
    lines.append("")
    lines.append("Factor exposures:")
    lines.append("")
    lines.append(df_to_markdown_safe(factor_exposure_report, max_rows=20))
    lines.append("")
    lines.append(f"![Factor exposure]({rel_chart_paths['factor_exposure']})")
    lines.append("")
    lines.append("## 7. Asset Attribution")
    lines.append("")
    lines.append("Top asset contributions:")
    lines.append("")
    lines.append(df_to_markdown_safe(asset_attribution, max_rows=15))
    lines.append("")
    lines.append(f"![Asset attribution]({rel_chart_paths['asset_attribution']})")
    lines.append("")
    lines.append("Asset-class contribution:")
    lines.append("")
    lines.append(df_to_markdown_safe(class_attribution, max_rows=10))
    lines.append("")
    lines.append("## 8. Turnover, Cost, and Capacity")
    lines.append("")
    lines.append("Cost / capacity summary:")
    lines.append("")
    lines.append(df_to_markdown_safe(cost_capacity_summary, max_rows=5))
    lines.append("")
    lines.append("Capacity by asset:")
    lines.append("")
    lines.append(df_to_markdown_safe(capacity_by_asset, max_rows=15))
    lines.append("")
    lines.append(f"![Capacity by asset]({rel_chart_paths['capacity_by_asset']})")
    lines.append("")
    lines.append("## 9. Regime and Stress Performance")
    lines.append("")
    lines.append(df_to_markdown_safe(regime_report, max_rows=20))
    lines.append("")
    lines.append("## 10. Validation Evidence")
    lines.append("")
    lines.append("Validation summary:")
    lines.append("")
    lines.append(df_to_markdown_safe(validation_summary, max_rows=5))
    lines.append("")
    lines.append("Validation notes:")
    lines.append("")
    lines.append(df_to_markdown_safe(validation_notes, max_rows=20))
    lines.append("")
    lines.append("Key validation fields:")
    lines.append("")
    lines.append("| Check | Value |")
    lines.append("|---|---:|")
    lines.append(f"| Walk-forward Sharpe | {format_num(val['walk_forward_sharpe'])} |")
    lines.append(f"| Bayesian P(alpha > 0) | {format_pct(val['bayesian_prob_positive_alpha'])} |")
    lines.append(f"| Deflated Sharpe probability | {format_pct(val['deflated_sharpe_probability'])} |")
    lines.append(f"| Probability of Backtest Overfitting | {format_pct(val['pbo'])} |")
    lines.append(f"| White Reality Check p-value | {format_pct(val['white_reality_check_pvalue'])} |")
    lines.append(f"| Purged CV AUC | {format_num(val['purged_cv_auc'])} |")
    lines.append(f"| Live paper Sharpe | {format_num(val['live_paper_sharpe'])} |")
    lines.append(f"| Live paper drawdown | {format_pct(val['live_paper_drawdown'])} |")
    lines.append("")
    lines.append("## 11. Governance Scorecard")
    lines.append("")
    lines.append(df_to_markdown_safe(governance_scorecard, max_rows=30))
    lines.append("")
    lines.append("Governance decision:")
    lines.append("")
    lines.append(df_to_markdown_safe(governance_decision, max_rows=5))
    lines.append("")
    lines.append("## 12. Input Validation and Audit")
    lines.append("")
    lines.append(df_to_markdown_safe(input_validation, max_rows=20))
    lines.append("")
    lines.append("## 13. Limitations")
    lines.append("")
    lines.append("1. This tearsheet uses synthetic data in this standalone notebook.")
    lines.append("2. Real deployment requires vendor data checks, execution data, broker reconciliation, and production logging.")
    lines.append("3. Transaction costs are simplified.")
    lines.append("4. Capacity is a proxy, not a full market-impact study.")
    lines.append("5. Validation metrics are simulated here, but in a real project they should be loaded from previous research notebooks.")
    lines.append("6. This report is not trading approval.")
    lines.append("")
    lines.append("## 14. Final Reviewer Questions")
    lines.append("")
    lines.append("1. Is the data clean and survivorship-safe?")
    lines.append("2. Are all failed experiments included in the research universe?")
    lines.append("3. Does the strategy pass purged validation?")
    lines.append("4. Does performance survive DSR, PBO, and Reality Check?")
    lines.append("5. Is cost drag acceptable?")
    lines.append("6. Is capacity sufficient?")
    lines.append("7. Did live paper monitoring behave like research?")
    lines.append("8. Are drawdowns acceptable?")
    lines.append("9. Are the main exposures intended?")
    lines.append("10. What would make us turn this strategy off?")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("Generated by `05_12_strategy_tearsheet_generator`.")

    return "\n".join(lines)

tearsheet_markdown = generate_markdown_tearsheet(
    config,
    performance_summary,
    drawdown_table,
    rolling_report,
    factor_exposure_report,
    factor_summary,
    asset_attribution,
    class_attribution,
    cost_capacity_summary,
    capacity_by_asset,
    regime_report,
    validation_summary,
    validation_notes,
    governance_scorecard,
    governance_decision,
    input_validation,
    chart_paths,
)

print(tearsheet_markdown[:1200])

## 20. Save tearsheet outputs

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)

tearsheet_path = output_dir / "strategy_tearsheet.md"
tearsheet_path.write_text(tearsheet_markdown, encoding="utf-8")

accounting.to_csv(output_dir / "accounting.csv")
asset_returns.to_csv(output_dir / "asset_returns.csv")
factor_returns.to_csv(output_dir / "factor_returns.csv")
held_weights.to_csv(output_dir / "held_weights.csv")
target_weights.to_csv(output_dir / "target_weights.csv")
delta_weights.to_csv(output_dir / "delta_weights.csv")
metadata.to_csv(output_dir / "metadata.csv", index=False)
regime_info.to_csv(output_dir / "regime_info.csv")

performance_summary.to_csv(output_dir / "performance_summary.csv", index=False)
drawdown_table.to_csv(output_dir / "drawdown_table.csv", index=False)
rolling_report.to_csv(output_dir / "rolling_report.csv")
factor_exposure_report.to_csv(output_dir / "factor_exposure_report.csv", index=False)
factor_summary.to_csv(output_dir / "factor_summary.csv", index=False)
factor_fitted.to_frame().to_csv(output_dir / "factor_fitted.csv")
factor_residual.to_frame().to_csv(output_dir / "factor_residual.csv")
asset_attribution.to_csv(output_dir / "asset_attribution.csv", index=False)
class_attribution.to_csv(output_dir / "class_attribution.csv", index=False)
cost_capacity_summary.to_csv(output_dir / "cost_capacity_summary.csv", index=False)
capacity_by_asset.to_csv(output_dir / "capacity_by_asset.csv", index=False)
regime_report.to_csv(output_dir / "regime_report.csv", index=False)
validation_summary.to_csv(output_dir / "validation_summary.csv", index=False)
validation_notes.to_csv(output_dir / "validation_notes.csv", index=False)
governance_scorecard.to_csv(output_dir / "governance_scorecard.csv", index=False)
governance_decision.to_csv(output_dir / "governance_decision.csv", index=False)
input_validation.to_csv(output_dir / "input_validation.csv", index=False)

tearsheet_path

## 21. Audit report

In [ ]:
audit_rows = []

audit_rows.append({
    "check": "tearsheet_markdown_exists",
    "value": str(tearsheet_path),
    "passed": bool(tearsheet_path.exists()),
})

for name, path in chart_paths.items():
    audit_rows.append({
        "check": f"chart_exists_{name}",
        "value": str(path),
        "passed": bool(Path(path).exists()),
    })

required_csvs = [
    "accounting.csv",
    "performance_summary.csv",
    "drawdown_table.csv",
    "factor_exposure_report.csv",
    "asset_attribution.csv",
    "governance_scorecard.csv",
    "governance_decision.csv",
]

for filename in required_csvs:
    path = output_dir / filename
    audit_rows.append({
        "check": f"output_exists_{filename}",
        "value": str(path),
        "passed": bool(path.exists()),
    })

audit_rows.append({
    "check": "governance_decision_present",
    "value": governance_decision.iloc[0]["overall_status"],
    "passed": bool(len(governance_decision) == 1),
})

audit_rows.append({
    "check": "tearsheet_does_not_approve_trading",
    "value": bool(governance_decision.iloc[0]["can_trade"] == False),
    "passed": bool(governance_decision.iloc[0]["can_trade"] == False),
})

audit_rows.append({
    "check": "input_validation_passed",
    "value": bool(input_validation["passed"].all()),
    "passed": bool(input_validation["passed"].all()),
})

perf_finite = bool(np.isfinite(performance_summary.select_dtypes(include=[float, int]).to_numpy()).all())
audit_rows.append({
    "check": "performance_metrics_finite",
    "value": perf_finite,
    "passed": perf_finite,
})

audit_report = pd.DataFrame(audit_rows)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

audit_report

## 22. Manifest export

In [ ]:
manifest = {
    "dataset_name": "strategy_tearsheet_generator_outputs",
    "schema_version": "strategy_tearsheet_generator_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "strategy_id": config.strategy_id,
    "strategy_name": config.strategy_name,
    "benchmark_name": config.benchmark_name,
    "output_dir": str(output_dir),
    "tearsheet_path": str(tearsheet_path),
    "chart_paths": {k: str(v) for k, v in chart_paths.items()},
    "headline_performance": performance_summary[performance_summary["series"] == "strategy"].to_dict(orient="records"),
    "benchmark_performance": performance_summary[performance_summary["series"] == "benchmark"].to_dict(orient="records"),
    "cost_capacity_summary": cost_capacity_summary.to_dict(orient="records"),
    "validation_summary": validation_summary.to_dict(orient="records"),
    "governance_decision": governance_decision.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook generates a strategy tearsheet from accounting, attribution, validation, risk and governance inputs.",
        "The standalone example uses synthetic data.",
        "The tearsheet is exported as Markdown with chart references.",
        "The governance decision is a research review status, not trading approval.",
        "The manifest records key output paths and headline metrics.",
        "This notebook closes Phase 5 and prepares the project for Phase 6 systems and microstructure notebooks."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

manifest_path

## 23. Practical checklist for a strategy tearsheet

A complete strategy tearsheet should include:

1. strategy ID and version;
2. data period and benchmark;
3. NAV and drawdown;
4. core performance metrics;
5. benchmark-relative alpha and beta;
6. factor attribution;
7. asset and class attribution;
8. turnover and cost drag;
9. capacity estimate;
10. validation evidence;
11. overfitting controls;
12. live paper summary;
13. governance scorecard;
14. limitations;
15. audit and manifest.

A strategy without a tearsheet is not ready for serious review.

## 24. Limitations

### Synthetic evidence pack

This notebook simulates all inputs for portability. In a real repo, it should load outputs from earlier notebooks or a research artifact store.

### Markdown export only

The notebook exports Markdown. A later extension could generate HTML, PDF, or a dashboard app.

### Simplified cost and capacity

Capacity is estimated with a proxy. Production capacity requires deeper impact modelling and execution data.

### Simulated validation metrics

The validation summary is simulated here. Real metrics should come from actual purged CV, PBO, DSR, WRC, Bayesian calibration, and live paper outputs.

### No formal approval workflow

The governance decision is a research signal, not a trading approval.

### No database or immutable storage

Outputs are CSV/JSON/Markdown files. Production needs versioned storage and immutable logs.

## 25. Research frontier and extensions

Important extensions include:

1. automatic HTML report generation;
2. PDF export;
3. dashboard app with Plotly or Streamlit;
4. report versioning;
5. dataset and code hashing;
6. experiment registry integration;
7. automated Slack/email distribution;
8. reviewer comments and sign-off workflow;
9. live report comparison versus backtest;
10. firm-wide standard strategy-tearsheet schema.

## 26. Phase 5 closing note

Phase 5 built a research hygiene stack:

1. vectorized backtest engine;
2. event-driven backtest engine;
3. transaction costs, slippage, and latency;
4. walk-forward optimisation;
5. Bayesian strategy calibration;
6. performance haircuts and Deflated Sharpe;
7. purged K-fold and embargo CV;
8. Probability of Backtest Overfitting;
9. White Reality Check bootstrap;
10. turnover, capacity, and alpha decay;
11. live paper dashboard without execution;
12. strategy tearsheet generator.

The phase turns strategy research from “nice backtest” into a reviewable research process.

## 27. Suggested Phase 6 direction

Phase 6 can now move into:

- execution systems;
- market microstructure;
- order book simulation;
- L1 bid/ask modelling;
- execution algorithms;
- market impact;
- latency;
- risk controls;
- production-style architecture.

The natural next phase is:

```text
Phase 6 — Execution, Market Microstructure & Systems
```

## 28. Summary

This notebook implemented a strategy tearsheet generator.

It showed how to:

1. simulate a complete strategy evidence pack;
2. validate input schemas;
3. compute performance metrics;
4. compute drawdown tables;
5. compute rolling metrics;
6. estimate factor attribution;
7. estimate asset and class attribution;
8. report cost, turnover, and capacity;
9. report regime/stress performance;
10. include validation evidence;
11. build a governance scorecard;
12. generate charts;
13. export a Markdown tearsheet;
14. save all supporting CSV outputs;
15. export a JSON manifest;
16. run final audit checks;
17. close Phase 5.

The key computational takeaway:

> A tearsheet generator is an artifact compiler: it turns raw research outputs into a reproducible review package.

The key financial takeaway:

> A strategy should not be judged by one equity curve; it should be judged by a complete evidence trail.

## 29. Further reading

- Grinold and Kahn, *Active Portfolio Management.*
- López de Prado, *Advances in Financial Machine Learning.*
- Bailey et al. on backtest overfitting.
- White, "A Reality Check for Data Snooping."
- Hansen, "A Test for Superior Predictive Ability."
- Institutional model-risk, investment committee, and research-governance documentation.